# Voorwaarden isolatiesubsidies van gemeentewebsites

Deze notebook zoekt per gemeente de webpagina's (en pdf's) over subsidie voor woningisolatie en laat
Gemini de voorwaarden uitlezen, in hetzelfde formaat als `regelingen.json`. Aanvulling op
`scraper/run.py`, dat alleen het CVDR doorzoekt.

**Draait op twee plekken**
- **Google Colab**: bestanden komen in Drive (`MyDrive/subsidie_checker`). Zet je Gemini-sleutel in
  Colab onder 🔑 *Secrets* als `GEMINI_API_KEY`.
- **GitHub Actions**: workflow *Webpagina's gemeenten scrapen* draait deze notebook; bestanden komen in
  `data/web/`. De sleutel komt uit het repo-secret `GEMINI_API_KEY`.

**Stappen**
1. Instellingen en hulpfuncties
2. Gemeentelijst (CBS)
3. Officiële websites (Register van Overheidsorganisaties)
4. Kandidaat-pagina's zoeken → `bronnen_sitemap.csv`
5. Voorwaarden uitlezen met Gemini → `voorwaarden.json`
6. Uitvoer → `regelingen_web.json`, `voorwaarden.csv`, `samenvatting.md`

Alles is **hervatbaar**: stopt de run (tijdslimiet, Gemini-limiet), dan gaat de volgende run verder waar
hij was. Wil je stap 4 helemaal opnieuw doen, zet dan `OPNIEUW = True`.

In [ ]:
# 1. Instellingen — werkt in Google Colab, in GitHub Actions en lokaal
import csv, datetime as dt, gzip, hashlib, io, json, os, pathlib, re, subprocess, sys, time, unicodedata
from collections import Counter
from urllib.parse import unquote, urljoin, urlparse

IN_COLAB = "google.colab" in sys.modules
IN_ACTIONS = os.environ.get("GITHUB_ACTIONS") == "true"
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf", "google-genai", "beautifulsoup4"], check=True)

import pandas as pd
import requests
from bs4 import BeautifulSoup

# --- Zelf aan te passen (in GitHub Actions overschreven door omgevingsvariabelen met dezelfde naam) ---
OPNIEUW = False              # True = bronnen_sitemap.csv opnieuw opbouwen (oude wordt bewaard als .oud)
FORCEER_EXTRACTIE = False    # True = ook ongewijzigde pagina's opnieuw door Gemini laten lezen
ALLEEN = []                  # bv. ["Apeldoorn", "Doesburg"] om te testen; leeg = alle gemeenten
MAX_GEMEENTEN_PER_RUN = 0    # 0 = geen maximum (telt per stap)
MAX_MINUTEN = 0              # 0 = geen maximum; daarna stopt de notebook netjes (GitHub: 6 uur per job)
MAX_MINUTEN_ZOEKEN = 0       # tijd voor stap 4, zodat er ook tijd overblijft voor stap 5
VERVERS_DAGEN = 7            # opgehaalde pagina's zo lang hergebruiken


def _env(naam, standaard):
    w = os.environ.get(naam)
    if w is None or w.strip() == "":
        return standaard
    if isinstance(standaard, bool):
        return w.strip().lower() in ("1", "true", "ja", "yes")
    if isinstance(standaard, int):
        return int(w)
    if isinstance(standaard, list):
        return [s.strip() for s in w.split(",") if s.strip()]
    return w


OPNIEUW = _env("OPNIEUW", OPNIEUW)
FORCEER_EXTRACTIE = _env("FORCEER_EXTRACTIE", FORCEER_EXTRACTIE)
ALLEEN = _env("ALLEEN", ALLEEN)
MAX_GEMEENTEN_PER_RUN = _env("MAX_GEMEENTEN_PER_RUN", MAX_GEMEENTEN_PER_RUN)
MAX_MINUTEN = _env("MAX_MINUTEN", MAX_MINUTEN)
MAX_MINUTEN_ZOEKEN = _env("MAX_MINUTEN_ZOEKEN", MAX_MINUTEN_ZOEKEN)
VERVERS_DAGEN = _env("VERVERS_DAGEN", VERVERS_DAGEN)


def zoek_repo():
    """De repo-map (met scraper/config.json), als de notebook vanuit de repo draait."""
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "scraper" / "config.json").exists():
            return p
    return None


ROOT = zoek_repo()
if IN_COLAB:
    from google.colab import drive, files, userdata
    drive.mount("/content/drive")
    MAP = "/content/drive/MyDrive/subsidie_checker"
else:
    MAP = os.environ.get("MAP") or str((ROOT or pathlib.Path.cwd()) / "data" / "web")
TEKSTEN = f"{MAP}/teksten"
os.makedirs(TEKSTEN, exist_ok=True)

REPO_RAW = "https://raw.githubusercontent.com/Martijn2412/Subsidie_Checker/main/"


def repo_bestand(pad):
    """Leest een bestand uit de repo: lokaal, anders uit de Drive-map, anders van GitHub (alleen bij een publieke repo)."""
    kandidaten = ([ROOT / pad] if ROOT else []) + [pathlib.Path(MAP) / pathlib.Path(pad).name]
    for p in kandidaten:
        if p.exists():
            return p.read_text(encoding="utf-8")
    try:
        r = requests.get(REPO_RAW + pad, timeout=30)
        if r.ok:
            return r.text
    except Exception:
        pass
    return None


CFG = json.loads(repo_bestand("scraper/config.json") or "{}")
MODELLEN = CFG.get("modellen", {}).get("gemini") or {"filter": "gemini-flash-lite-latest", "extractie": "gemini-flash-latest"}
PAUZE = CFG.get("pauze_tussen_aanroepen_sec", 7)  # gratis Gemini: max. enkele verzoeken per minuut
PROMPT = repo_bestand("scraper/prompt_extractie.md")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if IN_COLAB and not GEMINI_API_KEY:
    try:
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

UA = {"User-Agent": "Mozilla/5.0 (compatible; Takkenkamp-subsidiecheck/0.2; interne tool)"}
VANDAAG = dt.date.today().isoformat()
START = time.time()


def minuten():
    return (time.time() - START) / 60


def tijd_op(limiet=0):
    return bool((MAX_MINUTEN and minuten() > MAX_MINUTEN) or (limiet and minuten() > limiet))


print("Omgeving:", "Colab" if IN_COLAB else "GitHub Actions" if IN_ACTIONS else "lokaal")
print("Map:", MAP)
print("Prompt gevonden:", bool(PROMPT), "| Gemini-sleutel:", "ja" if GEMINI_API_KEY else "NEE (stap 5 wordt overgeslagen)")
if not PROMPT:
    print("  Zet scraper/prompt_extractie.md in", MAP, "(de repo is niet bereikbaar).")
print("Instellingen:", dict(OPNIEUW=OPNIEUW, ALLEEN=ALLEEN, MAX_GEMEENTEN_PER_RUN=MAX_GEMEENTEN_PER_RUN,
                            MAX_MINUTEN=MAX_MINUTEN, MAX_MINUTEN_ZOEKEN=MAX_MINUTEN_ZOEKEN))

In [ ]:
# Hulpfuncties: pagina ophalen (html of pdf) met cache, en tekst eruit halen
def naar_tekst(resp):
    """Geeft titel, platte tekst en links van een html-pagina of pdf."""
    ct = resp.headers.get("content-type", "").lower()
    if "pdf" in ct or resp.content[:4] == b"%PDF":
        from pypdf import PdfReader
        lezer = PdfReader(io.BytesIO(resp.content))
        tekst = "\n".join((p.extract_text() or "") for p in lezer.pages[:40])
        titel = (lezer.metadata.title if lezer.metadata else None) or unquote(urlparse(resp.url).path.rsplit("/", 1)[-1])
        links, pdf = [], True
    else:
        soup = BeautifulSoup(resp.content, "html.parser")
        titel = soup.title.get_text(" ", strip=True) if soup.title else ""
        links = []
        for a in soup.find_all("a", href=True):
            h = a["href"].split("#")[0].strip()
            if h and not h.lower().startswith(("mailto:", "tel:", "javascript:")):
                links.append([urljoin(resp.url, h), a.get_text(" ", strip=True)[:150]])
        for tag in soup(["script", "style", "noscript", "nav", "header", "footer", "aside", "form", "svg"]):
            tag.decompose()
        hoofd = soup.find("main") or soup.find("article") or soup.find(attrs={"role": "main"}) or soup.body or soup
        tekst, pdf = hoofd.get_text("\n", strip=True), False
    tekst = re.sub(r"\n\s*\n+", "\n", re.sub(r"[ \t\xa0]+", " ", tekst)).strip()
    return {"titel": titel[:300], "tekst": tekst[:100_000], "links": links, "pdf": pdf}


def pagina(url):
    """Haalt een pagina op; bewaart het resultaat VERVERS_DAGEN dagen in MAP/teksten."""
    pad = f"{TEKSTEN}/{hashlib.sha1(url.encode()).hexdigest()[:20]}.json"
    if os.path.exists(pad):
        try:
            p = json.load(open(pad, encoding="utf-8"))
            if (dt.date.today() - dt.date.fromisoformat(p["datum"])).days < VERVERS_DAGEN:
                return p
        except Exception:
            pass
    p = {"url": url, "datum": VANDAAG, "status": None, "titel": "", "tekst": "", "links": [], "pdf": False}
    try:
        x = requests.get(url, headers=UA, timeout=30, allow_redirects=True)
        p["status"], p["url_eind"] = x.status_code, x.url
        if x.status_code < 400:
            p.update(naar_tekst(x))
    except Exception as e:
        p["fout"] = str(e)[:200]
    if p["status"] is not None:  # netwerkfouten niet bewaren: volgende keer opnieuw proberen
        json.dump(p, open(pad, "w", encoding="utf-8"), ensure_ascii=False)
    time.sleep(1)  # netjes blijven tegen gemeentesites
    return p

In [ ]:
# 2. Gemeentelijst van het CBS (Gebieden in Nederland, meest recente jaar). Bewaard in gemeenten.json.
#    Liever je eigen lijst? Zet een gemeenten.json in de map: [{"naam": ..., "code": "GM0200", "provincie": ...}]
GEMEENTEN_JSON = f"{MAP}/gemeenten.json"
CBS = "https://opendata.cbs.nl/ODataApi/odata"


def cbs_json(url, **params):
    r = requests.get(url, params={"$format": "json", **params}, headers=UA, timeout=120)
    r.raise_for_status()
    return r.json()["value"]


def cbs_gemeenten():
    tabellen = []
    for t in cbs_json("https://opendata.cbs.nl/ODataCatalog/Tables", **{"$select": "Identifier,Title"}):
        m = re.fullmatch(r"Gebieden in Nederland (\d{4})", t["Title"].strip())
        if m and int(m.group(1)) <= dt.date.today().year:
            tabellen.append((int(m.group(1)), t["Identifier"]))
    jaar, tabel = max(tabellen)
    print(f"CBS-tabel {tabel} (Gebieden in Nederland {jaar})")
    rijen = cbs_json(f"{CBS}/{tabel}/TypedDataSet")
    props = {p.get("ID"): p for p in cbs_json(f"{CBS}/{tabel}/DataProperties")}
    naamkolommen = [p for p in props.values() if str(p.get("Key", "")).startswith("Naam_")]

    def kolom(groep, aantal):
        for p in naamkolommen:  # eerst via de groepsnaam in de metadata
            ouder = props.get(p.get("ParentID"), {})
            if re.match(groep, ouder.get("Title", ""), re.I):
                return p["Key"]
        for p in naamkolommen:  # anders via het aantal verschillende waarden
            if aantal(len({str(r.get(p["Key"])).strip() for r in rijen})):
                return p["Key"]
        raise RuntimeError(f"Kolom {groep} niet gevonden in {tabel}")

    k_naam = kolom(r"gemeente", lambda n: n == len(rijen))
    k_prov = kolom(r"provincie", lambda n: n == 12)
    return [{"naam": r[k_naam].strip(), "code": r["RegioS"].strip(), "provincie": r[k_prov].strip()} for r in rijen]


if os.path.exists(GEMEENTEN_JSON):
    gemeenten = json.load(open(GEMEENTEN_JSON, encoding="utf-8"))
else:
    gemeenten = cbs_gemeenten()
    json.dump(gemeenten, open(GEMEENTEN_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Gemeenten:", len(gemeenten))
if ALLEEN:
    onbekend = [n for n in ALLEEN if n not in {g["naam"] for g in gemeenten}]
    print("ALLEEN:", ALLEEN, "| onbekende namen:", onbekend or "geen")


def basis(naam):
    """'Bergen (NH.)' -> 'Bergen'"""
    return re.sub(r"\s*\([^)]*\)\s*$", "", naam).strip()


def selectie():
    return [g for g in gemeenten if not ALLEEN or g["naam"] in ALLEEN]

In [ ]:
# 3. Officiële websites uit het Register van Overheidsorganisaties (open data, CC0)
#    Koppeling via de CBS-code (gm0221) in de kolom 'TOOi URI'. Resultaat wordt bewaard in websites_roo.json.
CACHE = f"{MAP}/websites_roo.json"
ROO_CSV = "https://organisaties.overheid.nl/export/Gemeenten.csv"


def lees_roo(inhoud):
    roo = pd.read_csv(io.BytesIO(inhoud), sep=";", dtype=str, encoding="utf-8").fillna("")
    res = {}
    for _, rij in roo.iterrows():
        m = re.search(r"gm(\d{4})", rij.get("TOOi URI", "") + " " + rij.get("Organisatiecode", ""), re.I)
        if not m:
            continue
        sites = []
        for deel in rij.get("Internetpagina's", "").split(";"):
            url = deel.split(", label")[0].strip()
            if url.startswith("http"):
                u = urlparse(url)
                sites.append((0 if "algemeen" in deel else 1, f"{u.scheme}://{u.netloc.lower()}/"))
        if sites:
            res[m.group(1)] = list(dict.fromkeys(s for _, s in sorted(sites)))
    return res


if os.path.exists(CACHE):
    websites = json.load(open(CACHE))
else:
    try:
        r = requests.get(ROO_CSV, headers=UA, timeout=120)
        r.raise_for_status()
        websites = lees_roo(r.content)
    except Exception as e:
        print("Download lukt niet:", str(e)[:100])
        if not IN_COLAB:
            raise
        print("Download het bestand zelf via", ROO_CSV, "en upload het hieronder.")
        websites = lees_roo(next(iter(files.upload().values())))
    json.dump(websites, open(CACHE, "w"))


def cbs_nr(g):
    return re.sub(r"\D", "", g["code"]).zfill(4)


print("Websites uit het register:", sum(cbs_nr(g) in websites for g in gemeenten), "van", len(gemeenten), "gemeenten")
zonder = [g["naam"] for g in gemeenten if cbs_nr(g) not in websites]
print("Zonder website in het register (adres wordt gegokt):", zonder)


def gok_domein(naam):
    s = unicodedata.normalize("NFKD", basis(naam)).encode("ascii", "ignore").decode().lower()
    return f"https://www.{re.sub(r'[^a-z0-9-]', '', s.replace(' ', '-'))}.nl"


def site_van(g):
    kand = websites.get(cbs_nr(g), [])
    kand = sorted(kand, key=lambda u: "gemeente" not in u and basis(g["naam"]).lower()[:5] not in u)
    for u in kand + [gok_domein(g["naam"]), gok_domein(g["naam"]).replace("-", "")]:
        try:
            x = requests.get(u, headers=UA, timeout=20, allow_redirects=True)
            if x.status_code < 400:
                return x.url
        except Exception:
            pass
    return None

In [ ]:
# 4. Kandidaat-pagina's zoeken per gemeente → bronnen_sitemap.csv
#    Eerder gevonden gemeenten worden overgeslagen, behalve 'geen website gevonden' (die worden opnieuw geprobeerd).
UIT = f"{MAP}/bronnen_sitemap.csv"
KOLOMMEN = ["gemeente", "provincie", "url", "soort", "http_status", "noemt_isolatie", "gecontroleerd", "datum"]
OPNIEUW_PROBEREN = {"geen website gevonden"}

WOORDEN = {"isol": 4, "subsidie": 3, "verduurzam": 2, "duurzaam": 1, "energiebesp": 2, "lening": 1,
           "waardebon": 2, "energie": 1, "woning": 1, "spouw": 2, "glas": 1}
NIET = re.compile(r"/(nieuws|actueel|agenda|vergadering|raad|bekendmakingen|archief|vacatures?)(/|-|$)", re.I)
LOKET = re.compile(r"energieloket|duurzaambouwloket|regionaalenergieloket|woonwijzerwinkel|energiehuis|verbeterjehuis", re.I)
ZOEKPADEN = ["/zoeken?zoekterm=isolatie", "/zoeken?q=isolatie", "/search?q=isolatie", "/?s=isolatie"]


def score(tekst):
    t = tekst.lower()
    return sum(p for w, p in WOORDEN.items() if w in t)


def url_score(u, linktekst=""):
    pad = unquote(urlparse(u).path + " " + urlparse(u).query)
    return score(pad + " " + linktekst) - (5 if NIET.search(urlparse(u).path) else 0)


def sitemap_urls(site, max_sitemaps=25):
    basisurl = f"{urlparse(site).scheme}://{urlparse(site).netloc}"
    kaarten = []
    try:
        rob = requests.get(basisurl + "/robots.txt", headers=UA, timeout=20).text
        kaarten = re.findall(r"(?im)^sitemap:\s*(\S+)", rob)
    except Exception:
        pass
    kaarten = kaarten or [basisurl + "/sitemap.xml"]
    urls, gezien = [], 0
    while kaarten and gezien < max_sitemaps:
        k = kaarten.pop(0); gezien += 1
        try:
            x = requests.get(k, headers=UA, timeout=30)
            if x.status_code >= 400:
                continue
            inhoud = gzip.decompress(x.content).decode("utf-8", "ignore") if x.content[:2] == b"\x1f\x8b" else x.text
            locs = re.findall(r"<loc>\s*(?:<!\[CDATA\[)?\s*(.*?)\s*(?:\]\]>)?\s*</loc>", inhoud)
        except Exception:
            continue
        for l in locs:
            (kaarten if re.search(r"\.xml(\.gz)?$", l.lower().split("?")[0]) else urls).append(l)
    return urls


def op_site(u, host):
    return urlparse(u).netloc.lower().endswith(host)


def site_zoek(site, host):
    """Zoekfunctie van de site proberen als de sitemap weinig oplevert."""
    basisurl = f"{urlparse(site).scheme}://{urlparse(site).netloc}"
    for pad in ZOEKPADEN:
        p = pagina(basisurl + pad)
        gevonden = [u for u, t in p["links"] if op_site(u, host) and url_score(u, t) >= 4]
        if gevonden:
            return list(dict.fromkeys(gevonden))
    return []


def pagina_check(url):
    p = pagina(url)
    tekst = p["tekst"]
    loketten = {l for l, t in p["links"] if LOKET.search(l)}
    pdfs = {l for l, t in p["links"] if l.lower().split("?")[0].endswith(".pdf") and url_score(l, t) >= 4}
    return p["status"], score(tekst[:200_000]), "isol" in tekst.lower(), loketten, pdfs


def rij(g, url, soort, status="", isol=""):
    return {"gemeente": g["naam"], "provincie": g["provincie"], "url": url, "soort": soort,
            "http_status": status, "noemt_isolatie": isol, "gecontroleerd": "", "datum": VANDAAG}


def zoek_bronnen(g):
    site = site_van(g)
    if not site:
        return [rij(g, "", "geen website gevonden")], "GEEN WEBSITE"
    host = urlparse(site).netloc.lower().removeprefix("www.")
    urls = [(u, "") for u in sitemap_urls(site)] or [tuple(l) for l in pagina(site)["links"]]
    urls = list(dict.fromkeys((u, t) for u, t in urls if op_site(u, host)))
    kand = sorted({u: url_score(u, t) for u, t in urls if url_score(u, t) >= 4}.items(), key=lambda x: -x[1])
    top = [u for u, _ in kand[:6]]
    if len(top) < 3:
        top += [u for u in site_zoek(site, host) if u not in top][:6 - len(top)]
    treffers, loketten, pdfs = [], set(), set()
    for u in top:
        status, s, isol, lok, pdf = pagina_check(u)
        if isol:
            treffers.append((s, u, status)); loketten |= lok; pdfs |= pdf
    rijen = [rij(g, u, "gemeente", status, "ja") for s, u, status in sorted(treffers, reverse=True)[:3]]
    rijen += [rij(g, u, "pdf") for u in sorted(pdfs, key=lambda l: -url_score(l))[:3]]
    rijen += [rij(g, u, "loket") for u in sorted(loketten)[:2]]
    if not rijen:
        rijen = [rij(g, site, "alleen homepage, zelf zoeken")]
    return rijen, f"{len(urls)} pagina's op site, {len(treffers)} treffers, {len(pdfs)} pdf('s), {len(loketten)} loket(ten)"


def lees_bronnen():
    res = {}
    if os.path.exists(UIT):
        df = pd.read_csv(UIT, sep=";", dtype=str).fillna("")
        for r in df.to_dict("records"):
            res.setdefault(r["gemeente"], []).append({k: r.get(k, "") for k in KOLOMMEN})
    return res


def schrijf_bronnen(bronnen):
    volgorde = {g["naam"]: i for i, g in enumerate(gemeenten)}
    tmp = UIT + ".tmp"
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=KOLOMMEN, delimiter=";")
        w.writeheader()
        for naam in sorted(bronnen, key=lambda n: volgorde.get(n, 10**6)):
            w.writerows(bronnen[naam])
    os.replace(tmp, UIT)


if OPNIEUW and os.path.exists(UIT):
    os.replace(UIT, UIT + ".oud")
    print("Oude", UIT, "bewaard als .oud; begin opnieuw.")
bronnen = lees_bronnen()
if bronnen:
    print("Al in", UIT, ":", dict(Counter(r["soort"] for rijen in bronnen.values() for r in rijen)))
klaar = {n for n, rijen in bronnen.items() if not any(r["soort"] in OPNIEUW_PROBEREN for r in rijen)}
te_doen = [g for g in selectie() if g["naam"] not in klaar]
if MAX_GEMEENTEN_PER_RUN:
    te_doen = te_doen[:MAX_GEMEENTEN_PER_RUN]
print(f"{len(klaar)} gemeenten al klaar, nu te doen: {len(te_doen)}. (Alles opnieuw? Zet OPNIEUW = True.)")

for i, g in enumerate(te_doen):
    if tijd_op(MAX_MINUTEN_ZOEKEN):
        print(f"Tijd op na {minuten():.0f} min; {len(te_doen) - i} gemeenten volgen bij de volgende run.")
        break
    try:
        rijen, melding = zoek_bronnen(g)
    except Exception as e:
        rijen, melding = [rij(g, "", "geen website gevonden")], f"FOUT {str(e)[:100]}"
    bronnen[g["naam"]] = rijen
    schrijf_bronnen(bronnen)
    print(i, g["naam"], melding)
print("Klaar:", UIT, "|", dict(Counter(r["soort"] for rijen in bronnen.values() for r in rijen)))

In [ ]:
# 5. Voorwaarden uitlezen met Gemini (zelfde prompt en modellen als scraper/run.py) → voorwaarden.json
#    Per gemeente worden de gevonden pagina's samengevoegd. Alleen gewijzigde teksten gaan opnieuw naar Gemini.
VW = f"{MAP}/voorwaarden.json"
voorwaarden = json.load(open(VW, encoding="utf-8")) if os.path.exists(VW) else {}
BRUIKBAAR = ("gemeente", "pdf", "loket")

client = None
if GEMINI_API_KEY and PROMPT:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY, http_options=types.HttpOptions(timeout=180_000))


class LimietOp(Exception):
    """De (gratis) limiet van het taalmodel is bereikt; de rest volgt bij de volgende run."""


def llm(taak, tekst, json_uit=False):
    """Zelfde logica als scraper/run.py: bij drukte wachten, bij limiet stoppen."""
    modellen = [MODELLEN[taak]] + ([MODELLEN[taak + "_reserve"]] if MODELLEN.get(taak + "_reserve") else [])
    laatste_fout = None
    for model in modellen:
        for poging in range(3):
            try:
                time.sleep(PAUZE)
                cfg = types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=16000 if taak == "extractie" else 1000,
                    response_mime_type="application/json" if json_uit else "text/plain",
                )
                r = client.models.generate_content(model=model, contents=tekst, config=cfg)
                if not r.text:
                    raise RuntimeError("leeg antwoord")
                return r.text
            except Exception as e:
                fout = str(e)
                laatste_fout = fout
                limiet = "429" in fout or "RESOURCE_EXHAUSTED" in fout
                if limiet and poging >= 1:
                    print(f"  {model}: limiet bereikt, ander model proberen")
                    break
                wacht = 40 if limiet else 20 * (poging + 1)
                print(f"  {model}: fout ({fout[:110]}), wacht {wacht}s, poging {poging + 1}/3")
                time.sleep(wacht)
    if laatste_fout and ("429" in laatste_fout or "RESOURCE_EXHAUSTED" in laatste_fout):
        raise LimietOp(laatste_fout[:200])
    raise RuntimeError(f"Taalmodel faalt: {(laatste_fout or '')[:200]}")


def parse_json(tekst):
    tekst = re.sub(r"^```(?:json)?|```$", "", tekst.strip(), flags=re.M).strip()
    return json.loads(tekst)


def tekst_valt_af(tekst):
    """Goedkope trechter: moet over isolatie van woningen én geld gaan."""
    laag = tekst.lower()
    return not ("isol" in laag and ("woning" in laag or "eigenaar" in laag or "huis" in laag)
                and re.search(r"subsidie|lening|tegoed|voucher|waardebon|bijdrage", laag))


def is_relevant(naam, tekst):
    vraag = ("Beantwoord met alleen JA of NEE.\n"
             f"Hieronder staan webpagina's van gemeente {naam} en/of het energieloket.\n"
             f"JA als ze een regeling beschrijven waarmee inwoners van {naam} geld krijgen (subsidie, lening, "
             "waardebon, voucher) voor isolatie van hun BESTAANDE woning: dak, zolder, gevel, spouwmuur, "
             "vloer/bodem of isolerend glas.\n"
             "NEE als het alleen gaat over de landelijke ISDE, energieadvies of -coaches zonder geld, "
             "aardgasvrij/warmtepompen/zonnepanelen zonder isolatie, of monumenten, bedrijven of verhuurders.\n\n"
             f"{tekst[:8000]}")
    return llm("filter", vraag).strip().upper().startswith("JA")


def extraheer(naam, tekst):
    intro = (f"LET OP: de tekst hieronder komt van webpagina's (en eventueel pdf's) van gemeente {naam} of het "
             "energieloket, niet uit het officiële regelingenregister. Neem alleen voorwaarden over die gelden "
             f"voor inwoners van gemeente {naam}. Beschrijven de bronnen meerdere regelingen, neem dan de regeling "
             "die het meest direct isolatie van de eigen woning betaalt en noem de andere kort in \"opmerkingen\".")
    antw = llm("extractie", PROMPT + "\n\n" + intro + "\n\n<regelingstekst>\n" + tekst + "\n</regelingstekst>", json_uit=True)
    return parse_json(antw)


def bronnen_tekst(rijen):
    """Haalt de pagina's op en voegt de relevante samen (max. 60.000 tekens)."""
    stukken, urls = [], []
    for r in rijen:
        p = pagina(r["url"])
        if p["tekst"] and not tekst_valt_af(p["tekst"]):
            stukken.append(f"### Bron ({r['soort']}): {r['url']}\n### Titel: {p['titel']}\n{p['tekst'][:25_000]}")
            urls.append(r["url"])
    return "\n\n".join(stukken)[:60_000], urls


def bewaar(naam, **velden):
    voorwaarden[naam] = {"datum": VANDAAG, **velden}
    tmp = VW + ".tmp"
    json.dump(voorwaarden, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    os.replace(tmp, VW)


tel, gedaan = Counter(), 0
for g in selectie():
    naam = g["naam"]
    if tijd_op():
        print(f"Tijd op na {minuten():.0f} min; de rest volgt bij de volgende run."); break
    if MAX_GEMEENTEN_PER_RUN and gedaan >= MAX_GEMEENTEN_PER_RUN:
        print("MAX_GEMEENTEN_PER_RUN bereikt."); break
    if naam not in bronnen:
        tel["nog niet gezocht"] += 1; continue
    rijen = [r for r in bronnen[naam] if r["soort"] in BRUIKBAAR and r["url"]]
    if not rijen:
        bewaar(naam, uitkomst=bronnen[naam][0]["soort"]); tel[bronnen[naam][0]["soort"]] += 1; continue
    vorig = voorwaarden.get(naam, {})
    if not FORCEER_EXTRACTIE and vorig.get("hash") and vorig.get("bron_datum") == VANDAAG:
        tel["vandaag al gedaan"] += 1; continue
    tekst, urls = bronnen_tekst(rijen)
    if not tekst:
        bewaar(naam, uitkomst="geen isolatieregeling op website", bronnen=[r["url"] for r in rijen])
        tel["geen isolatieregeling op website"] += 1; continue
    h = hashlib.sha256(tekst.encode()).hexdigest()
    if not FORCEER_EXTRACTIE and vorig.get("hash") == h:
        tel["ongewijzigd"] += 1; continue
    if not client:
        tel["wacht op Gemini-sleutel"] += 1; continue
    gedaan += 1
    try:
        print(f"{naam}: Gemini-filter ({len(urls)} bron(nen), {len(tekst)} tekens)")
        if not is_relevant(naam, tekst):
            bewaar(naam, uitkomst="niet relevant (Gemini-filter)", hash=h, bron_datum=VANDAAG, bronnen=urls)
            tel["niet relevant"] += 1; continue
        print(f"{naam}: uitlezen")
        ext = extraheer(naam, tekst)
    except LimietOp as e:
        print(f"LIMIET BEREIKT ({e}). De rest volgt bij de volgende run.")
        tel["uitgesteld (limiet)"] += 1; break
    except Exception as e:
        print(f"{naam}: mislukt ({str(e)[:150]})")
        tel["mislukt"] += 1; continue
    if not ext.get("relevant", True):
        bewaar(naam, uitkomst="niet relevant (extractie)", hash=h, bron_datum=VANDAAG, bronnen=urls)
        tel["niet relevant"] += 1; continue
    ext.pop("relevant", None)
    bewaar(naam, uitkomst="gevonden", hash=h, bron_datum=VANDAAG, bronnen=urls, regeling=ext)
    tel["gevonden"] += 1
    print(f"{naam}: {ext.get('naam')} | {ext.get('bedrag')}")
print("Stap 5:", dict(tel))

In [ ]:
# 6. Uitvoer: regelingen_web.json (formaat van regelingen.json), voorwaarden.csv en samenvatting.md
def slug(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")


def status(rec):
    eind = rec.get("looptijd_eind")
    if eind and eind < VANDAAG:
        return "gesloten"
    return "open" if eind else "onbekend"


def betrouwbaarheid(rec):
    """Websites zijn geen officiële regelingstekst: hooguit 'middel'."""
    if re.search(r"tegenstrijd|spreekt.*tegen", rec.get("opmerkingen") or "", re.I):
        return "laag"
    return "middel" if rec.get("bedrag") and rec.get("looptijd_eind") else "laag"


prov = {g["naam"]: g["provincie"] for g in gemeenten}
cvdr = set()
if ROOT and (ROOT / "regelingen.json").exists():
    cvdr = {r["gemeente"] for r in json.loads((ROOT / "regelingen.json").read_text(encoding="utf-8"))}

regelingen, overzicht = [], []
for g in gemeenten:
    naam = g["naam"]
    v = voorwaarden.get(naam)
    ext = (v or {}).get("regeling")
    if ext:
        rec = {"id": f"{slug(naam)}-web", "cvdr_id": None, "versie": None, "handmatig": False, "gecontroleerd": False,
               "gemeente": naam, "provincie": prov.get(naam), **ext,
               "bron_url": v["bronnen"][0] if v.get("bronnen") else None, "bronnen": v.get("bronnen", []),
               "brontype": "website", "datum_regelingstekst": None, "peildatum": v.get("bron_datum")}
        rec["naam"] = rec.get("naam") or f"Isolatieregeling {naam} (website)"
        rec["status"] = status(rec)
        rec["betrouwbaarheid"] = betrouwbaarheid(rec)
        regelingen.append(rec)
    c = (ext or {}).get("criteria") or {}
    overzicht.append({
        "gemeente": naam, "provincie": g["provincie"],
        "uitkomst": (v or {}).get("uitkomst", "nog niet gezocht" if naam not in bronnen else "nog niet uitgelezen"),
        "regeling": (ext or {}).get("naam"), "bedrag": (ext or {}).get("bedrag"),
        "woz_max": c.get("woz_max"), "bouwjaar_max": c.get("bouwjaar_max"), "inkomen": c.get("inkomen"),
        "looptijd_eind": (ext or {}).get("looptijd_eind"), "status": status(ext) if ext else None,
        "ook_in_cvdr": "ja" if naam in cvdr else "",
        "bron": ((v or {}).get("bronnen") or [""])[0],
    })

json.dump(regelingen, open(f"{MAP}/regelingen_web.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
df = pd.DataFrame(overzicht)
df.to_csv(f"{MAP}/voorwaarden.csv", sep=";", index=False, encoding="utf-8")

tellingen = df["uitkomst"].value_counts()
nieuw_web = [r for r in regelingen if r["gemeente"] not in cvdr]
md = [f"# Webpagina's gemeenten {VANDAAG}", "",
      f"{len(regelingen)} regelingen gevonden op websites ({len(nieuw_web)} bij gemeenten zonder CVDR-regeling in regelingen.json).", "",
      "| uitkomst | gemeenten |", "|---|---|", *[f"| {k} | {n} |" for k, n in tellingen.items()], ""]
if regelingen:
    md += ["## Gevonden", "", "| gemeente | regeling | bedrag | eind | status | bron |", "|---|---|---|---|---|---|"]
    cel = lambda x: str(x or "").replace("|", "/").replace("\n", " ")
    md += [f"| {cel(r['gemeente'])} | {cel(r['naam'])} | {cel(r.get('bedrag'))} | {cel(r.get('looptijd_eind'))} | {r['status']} | {cel(r['bron_url'])} |"
           for r in regelingen]
md += ["", "Controleer elke regeling tegen de bron voordat je hem overneemt in regelingen.json (`\"gecontroleerd\": true`)."]
open(f"{MAP}/samenvatting.md", "w", encoding="utf-8").write("\n".join(md) + "\n")

print(tellingen.to_string())
print(f"\nGeschreven in {MAP}: regelingen_web.json ({len(regelingen)}), voorwaarden.csv, samenvatting.md")
df[df["uitkomst"] == "gevonden"].head(30)